*Insert here your name*

The first part of this notebook is based on Sudalai Rajkumar's tutorial on Kaggle.
More information on the [***dataset***](https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter).


## **Introduction**

In any machine learning task, cleaning or preprocessing the data is as important as model building if not more. And when it comes to unstructured data like text, this process is even more important.

The goal of this tutorial is to understand the various text preprocessing steps with code examples.

Some of the common text preprocessing / cleaning steps are:
* Lower casing
* Removal of Punctuations
* Removal of Stopwords
* Removal of Frequent words
* Removal of Rare words
* Stemming
* Lemmatization
* Removal of emojis
* Removal of emoticons
* Conversion of emoticons to words
* Conversion of emojis to words
* Removal of URLs
* Removal of HTML tags
* Chat words conversion
* Spelling correction


So these are the different types of text preprocessing steps which we can do on text data. But we need not do all of these all the times. We need to carefully choose the preprocessing steps based on our use case since that also play an important role.

For example, in sentiment analysis use case, we need not remove the emojis or emoticons as they will convey some important information about the sentiment.

In [ ]:
import numpy as np
import pandas as pd
import re
import nltk
import spacy
import string
pd.options.mode.chained_assignment = None

Let's load the dataset and see how is structured with few samples

In [ ]:
df = pd.read_csv("./data/tweets_preprocessing.csv")
df["text"] = df["text"].astype(str)
print(f"Shape: {df.shape}")
df.head()

## **Lower Casing**

Lower casing is a common text preprocessing technique. The idea is to convert the input text into same casing format so that 'text', 'Text' and 'TEXT' are treated the same way.

This is more helpful when computing words frequency for example, yet it may not be the case for tasks like Part of Speech tagging (where proper casing gives some information about proper nouns and so on) and Sentiment Analysis (where upper casing refers to anger)

By default, lower casing is done my most of the modern day vectorizers and tokenizers.

In [ ]:
text_df = df[["text"]]

In [ ]:
print("Before lowering:")
print(text_df.head().text.values)

In [ ]:
# Your code here:
# convert the text_df text column to lowercase
# keep in mind that text_df["text"] is a Pandas' Series

assert text_df.text.values[2] == "@76328 i really hope you all change but i'm sure you won't! because you don't have to!", "Your text is not lowered properly"

print("After lowering:")
print(text_df.head().text.values)
text_df.head()

## **Removal of Punctuations**

Another common text preprocessing technique is to remove the punctuations from the text data. This is again a text standardization process that will help to treat 'hurray' and 'hurray!' in the same way.

We also need to carefully choose the list of punctuations to exclude depending on the use case. For example, the `string.punctuation` in python contains the following punctuation symbols

`!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~`

We can add or remove more punctuations as per our need.

In [ ]:
string.punctuation

In [ ]:
PUNCT_TO_REMOVE = string.punctuation
def remove_punctuation(text: str) -> str:
    """
    Removes punctuation characters from the input text

    Args:
        text (str): The input text from which punctuation characters will be removed

    Returns:
        str: A new string with all punctuation characters removed
    """
    # Your code here:
    # use the maketrans function to remove the punctuation specified in PUNCT_TO_REMOVE [https://www.w3schools.com/python/ref_string_maketrans.asp]

# now apply your function to the text column using the apply function [https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html]

assert text_df["text_wo_punct"].values[3] == "105836 livechat is online at the moment  httpstcosy94vtu8kq or contact 03331 031 031 option 1 4 3 leave a message to request a call back"
text_df.head()

## **Removal of stopwords**

Stopwords are commonly occuring words in a language like 'the', 'a' and so on. They can be removed from the text most of the times, as they don't provide valuable information for downstream analysis. In cases like Part of Speech tagging, we should not remove them as provide very valuable information about the POS.

These stopword lists are already compiled for different languages and we can safely use them. For example, the stopword list for english language from the nltk package can be seen below.


In [ ]:
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))
", ".join(stopwords.words('english'))

Similarly we can also get the list for other languages as well and use them.

In [ ]:
sample = text_df.text.values[0]
sample

In [ ]:
split = sample.split(' ') # splits the input string into a list, the delimiter is a whitespace " "
split

In [ ]:
# Use list comprehension to remove the stopwords: [https://www.programiz.com/python-programming/list-comprehension]
# We want to achieve the same result as:

# filtered_words = []
# for word in split:
#   if word not in STOPWORDS:
#       filtered_words.append(word)
#
# Your code here


assert filtered_words == ['@applesupport','causing','reply','disregarded','tapped','notification','keyboard','opened😡😡😡']
filtered_words

In [ ]:
filtered_string = " ".join(filtered_words)
print(f"Before filtering: {sample}\nAfter filtering: {filtered_string}")

In [ ]:
def remove_stopwords(text: str) -> str:
    """
    Removes stopwords from the input text

    Args:
        text (str): The input text from which stopwords will be removed

    Returns:
        str: A new string without stopwords
    """
    # Your code here:
    # remove stopwords with list comprehension and convert the list back to a string by concatenating the words

text_df["text_wo_stop"] = text_df["text_wo_punct"].apply(remove_stopwords)
text_df.head()

## **Removal of Frequent words**

In the previous preprocessing step, we removed the stopwords based on language information. But say, if we have a domain specific corpus, we might also have some frequent words which are of lesser importance to us.

So this step is to remove the frequent words in the given corpus. So, let us get the most common words and then remove them in the next step

In [ ]:
from collections import Counter
cnt = Counter()

# Your code here:
# Use the Counter class to return the most frequent words
# 1: join all the text in the text_wo_stop column using the join() function
# 2: tokenize the text on white space by using the split() function
# 3: instantiate the Counter class with your tokenized array
# 4: use the most_common class method to return the most frequent words

assert set(most_common[:10]) == set([('us', 25), ('dm', 19),('help', 18),('thanks', 13),('httpstcogdrqu22ypt',12),('applesupport', 11),('please', 11),('phone', 9),('hi', 9),('ive', 8)])
most_common[:10]

In [ ]:
# create a set with the 10 most frequent words:
# hint: create a set comprehension equivalent to
# words = set()
# for word, count in most_common[:10]:
#   words.add(word)
#
# keep in mind that sets are more efficient in this scenario, being implemented as hash tables

FREQWORDS = {w for (w, word_count) in most_common[:10]}

def remove_freqwords(text: str, freq_words: set=FREQWORDS) -> str:
    """
    Removes a selection of frequent words from the input string

    Inputs:
        text (str): The input text from which frequent words will be removed
        freq_words (set): A set of frequent words to remove from the text

    Returns:
        str: A new string with all frequent words removed
    """
    # Your code here:
    # use your function to filter out the 10 most frequent words

assert remove_freqwords(text_df.text_wo_stop.values[1]) == "105835 business means lot name zip code additional details concern rr httpstcoznuu1vjn9r"

# Your code here:
# Apply your function to the text_wo_stop column

text_df.head()

## **Removal of Rare words**

This is very similar to previous preprocessing step but we will remove the rare words from the corpus.

In [ ]:
# let's keep only the latest version of the processed text
text_df = text_df[["text", "text_wo_stopfreq"]]
text_df.head()

,text,text_wo_stopfreq
0,@applesupport causing the reply to be disregar...,applesupport causing reply disregarded tapped ...
1,@105835 your business means a lot to us. pleas...,105835 business means lot name zip code additi...
2,@76328 i really hope you all change but i'm su...,76328 really hope change im sure wont dont
3,@105836 livechat is online at the moment - htt...,105836 livechat online moment httpstcosy94vtu8...
4,@virgintrains see attached error message. i've...,virgintrains see attached error message tried ...


In [ ]:
# Your code here:
# similarly to FREQWORD, extract the 10 rarest words

assert RAREWORDS == set(['browser', 'green', 'httpstco9281okeebk', 'including', 'keen', 'lee', 'line', 'log','slowdown','thin'])
RAREWORDS

In [ ]:
def remove_rarewords(text: str, rare_words: set=RAREWORDS) -> str:
    """
    Removes a selection of rare words from the input string

    Inputs:
        text (str): The input text from which rare words will be removed
        rare_words (set): A set of rare words to remove from the text

    Returns:
        str: A new string with all rare words removed
    """
    # Your code here:
    # Filter out the most frequent words from a text string

text_df["text_wo_stopfreqrare"] = text_df["text_wo_stopfreq"].apply(remove_rarewords)
text_df.head()

We can combine all the list of words (stopwords, frequent words and rare words) and create a single list to remove them at once.

In [ ]:
# Your code here:
# group all the words to remove using in a single set (TO_REMOVE)


In [ ]:
def filter_text(text: str, words_to_remove :set=TO_REMOVE) -> str:
    """
    Removes a selection of words from the input string

    Inputs:
        text (str): The input text from which words will be removed
        words_to_remove (set): A set of words to remove from the text

    Returns:
        str: A new string with all listed words removed
    """
    # Your code here

text_df["filtered_text"] =  text_df.text.apply(filter_text)
text_df = text_df[['text', 'filtered_text']]
text_df.head()

## **Stemming**

Stemming is the process of reducing inflected (or sometimes derived) words to their word stem, base or root form (From [Wikipedia](https://en.wikipedia.org/wiki/Stemming)).

This process is useful to **`reduce the vocabulary size`** by converting similar words to their root form.

For example, if there are two words in the corpus `walks` and `walking`, then stemming will stem the suffix to make them `walk`. But say in another example, we have two words `console` and `consoling`, the stemmer will remove the suffix and make them `consol` which is not a proper english word.

There are several type of stemming algorithms available and one of the famous one is porter stemmer (NLTK package) which is widely used.

In [ ]:
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()

def stem_demonstration(word: str) -> None:
    """
    Compares a word before and after stemming
    """
    print(f"Before stemming: {word}, \tafter stemming: {stemmer.stem(word)}")

for word in ['programs', 'programming', 'programmed', 'walks', 'walked', 'walking', 'UPPERCASE', 'mistake', 'mistakke']:
    stem_demonstration(word)

In [ ]:
def stem_words(text: str) -> str:
    """
    Applies the stemmer to an input string

    Args:
        text (str): The text to be stemmed

    Returns:
        str: A new string where every word has been stemmed
    """
    # Your code here:

assert stem_words(text_df["filtered_text"].values[7]) == "@105836 work ok here, miriam. link https://t.co/0m2mph15eh ? ^mm" # note that the stemmer also applied lowercase

text_df["text_stemmed"] = text_df["filtered_text"].apply(stem_words)
text_df.head()

In [ ]:
all_text_no_stemming = ' '.join(text_df["text"]).split()
all_text_w_stemming = ' '.join(text_df["text_stemmed"]).split()

n_words_no_stemming = len(set(all_text_no_stemming))
n_words_w_stemming = len(set(all_text_w_stemming))
vocabulary_size_diff = n_words_no_stemming - n_words_w_stemming

assert vocabulary_size_diff == 156

print(f"Number of unique words without stemming: {n_words_no_stemming}")
print(f"Number of unique words with stemming: {n_words_w_stemming}")
print(f"Difference: {vocabulary_size_diff} words")

We can see that words like `private` and `propose` have their `e` at the end chopped off due to stemming. This is not intented. What can we do for that? We can use Lemmatization in such cases.

This porter stemmer is for English language only, if we are working with other languages, we can use the snowball stemmer. The supported languages for snowball stemmer are:

In [ ]:
from nltk.stem.snowball import SnowballStemmer
SnowballStemmer.languages

## **Lemmatization**

Lemmatization is similar to stemming in reducing inflected words to their word stem but differs in the way that it makes sure the root word (also called as lemma) belongs to the language.
Here's a list of [examples](https://github.com/michmech/lemmatization-lists/blob/master/lemmatization-en.txt).

As a result, this one is generally slower than stemming process. So depending on the speed requirement, we can choose to use either stemming or lemmatization.

Let us use the `WordNetLemmatizer` in nltk to lemmatize our sentences

In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

In [ ]:
lemmatizer = WordNetLemmatizer()

def lem_demonstration(word: str, pos: str='n') -> None:
    print(f"Before lemmatization: {word}\tafter lemmatization: {lemmatizer.lemmatize(word, pos)}")

for word, pos in [('feet', 'n'), ('caring', 'v')]:
    lem_demonstration(word, pos)

In [ ]:
def lemmatize_words(text: str) -> str:
    """
    Applies lemmatization to the input string

    Args:
        text (str): The input text to lemmatize

    Returns:
        str: The lemmatized version of the text
    """
    # Your code here

text_df["text_lemmatized"] = text_df["filtered_text"].apply(lemmatize_words)
text_df.head()

We can see that the trailing `e` in the `propose` and `private` is retained when we use lemmatization unlike stemming.

Wait. There is one more thing in lemmatization. Let us try to lemmatize `running` now.

In [ ]:
lemmatizer.lemmatize("running")

Wow. It returned `running` as such without converting it to the root form `run`. This is because the lemmatization process depends on the POS tag to come up with the correct lemma. Now let us lemmatize again by providing the POS tag for the word.

See this [table](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html) for examples of POS tags.

In [ ]:
lemmatizer.lemmatize("running", "v") # v for verb

Now we are getting the root form `run`. So we also need to provide the POS tag of the word along with the word for lemmatizer in nltk. Depending on the POS, the lemmatizer may return different results.

Let us take the example, `stripes` and check the lemma when it is both verb and noun.

In [ ]:
print("Word is: stripes")
print("Lemma result for verb: ",lemmatizer.lemmatize("stripes", 'v'))
print("Lemma result for noun: ",lemmatizer.lemmatize("stripes", 'n'))

Now let us redo the lemmatization process for our dataset.

In [ ]:
from nltk.corpus import wordnet
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
lemmatizer = WordNetLemmatizer()
def lemmatize_words(text: str) -> str:
    """
    Apply lemmatization to the input string, considering words' POS tags.

    This function lemmatizes words in the input string based on their POS (Part-of-Speech) tags.

    Args:
        text (str): The input text to be lemmatized.

    Returns:
        str: A new string with lemmatized words.
    """

    # Initialize a mapping of POS tags to WordNet tags
    wordnet_map = {
        'N': wordnet.NOUN,
        'V': wordnet.VERB,
        'R': wordnet.ADV,
        'J': wordnet.ADJ
    }

    # Your code here:
    # Use the nltk.pos_tag function to get the POS tags of every word in the input (https://www.nltk.org/api/nltk.tag.pos_tag.html)
    # You may also use nltk.word_tokenize to tokenize the text (https://www.nltk.org/api/nltk.tokenize.html)

    # Your code here:
    # Return the lemmatized version of the text
    # Inside the lemmatize function, use the (word, POS tag) tuple collected in the pos_tagged_text list
    # hint: query the wordnet_map with wordnet_map.get(key, default) using wordnet.NOUN as default


assert lemmatize_words("Any man who must say 'I am the king' is no true king.") == "Any man who must say ' I be the king ' be no true king ."
text_df["text_lemmatized"] = text_df["filtered_text"].apply(lambda text: lemmatize_words(text))

text_df.head()

In [ ]:
all_text_no_lemm = ' '.join(text_df["text"]).split()
all_text_w_lemm = ' '.join(text_df["text_lemmatized"]).split()

n_words_no_lemm = len(set(all_text_no_lemm))
n_words_w_lemm = len(set(all_text_w_lemm))
vocabulary_size_diff = n_words_no_lemm - n_words_w_lemm

assert vocabulary_size_diff == 221

print(f"Number of unique words without stemming: {n_words_no_lemm}")
print(f"Number of unique words with stemming: {n_words_w_lemm}")
print(f"Difference: {vocabulary_size_diff} words out of {df.shape[0]} sample")

## **Removal of URLs**

Next preprocessing step is to remove any URLs present in the data. For example, if we are doing a twitter analysis, then there is a good chance that the tweet will have some URL in it. Probably we might need to remove them for our further analysis.

We can use the below code snippet to do that.

`Regex breakdown:`
```Python
r'https?://\S+|www\.\S+'
# could also be understood as
(r'https?://\S+') or (r'www\.\S+')
```
* `r` in front of a string indicates that Python shall treat the string as a raw litteral (avoids `\` being treated as escape characters)
* `https?://'`: This part of the regular expression matches URLs that start with either "http://" or "https:////". The `s?` portion allows for an optional "s" character, so it matches both "http://" and "https://".
*  `\S+`: This part of the regular expression matches one or more non-whitespace characters. It's used to match the domain part of the URL (e.g., www.example.com).
|: This is the alternation operator, which acts like an OR operator in regular expressions. It allows you to match either the pattern on the left or the pattern on the right. In this case, it's used to match either URLs starting with "http://" or "https://", or URLs starting with "www.".
* `www\.\S+`: This part of the regular expression matches URLs that start with "www." and then followed by one or more non-whitespace characters. It's commonly used to match URLs like "www.example.com".

In summary, this regular expression is designed to identify and capture URLs in a text string, whether they start with `"http://"`, `"https://"`, or `"www."`. It's a common pattern for extracting or hyperlinking URLs in text processing tasks.
So, when you see `u'('+emot+')'`, it's creating a Unicode string that contains a left parenthesis `'('`, the value of the `emot` variable (which is a placeholder for the text or pattern you want to find), and a right parenthesis `')'`.

In [ ]:
def remove_urls(text :str) -> str:
    """
    Remove URLs (web links) from the input text.

    This function searches for URLs in the input text and removes them, leaving the
    text without any web links.

    Args:
        text (str): The input text from which URLs will be removed.

    Returns:
        str: A new string with URLs removed.

    Example:
        >>> remove_urls("Visit our website at https://www.example.com to learn more.")
        "Visit our website at to learn more."
    """
    # Your code here

Let us take a `https` link and check the code

In [ ]:
text = "Driverless AI NLP blog post on https://www.h2o.ai/blog/detecting-sarcasm-is-difficult-but-ai-may-have-an-answer/"
remove_urls(text)

Now let us take a `http` url and check the code

In [ ]:
text = "Please refer to link http://lnkd.in/ecnt5yC for the paper"
remove_urls(text)

Thanks to Pranjal for the edge cases in the comments below. Suppose say there is no `http` or `https` in the url link. The function can now captures that as well.

In [ ]:
text = "Want to know more. Checkout www.h2o.ai for additional information"
remove_urls(text)

## **Removal of HTML Tags**

One another common preprocessing technique that will come handy in multiple places is removal of html tags. This is especially useful, if we scrap the data from different websites. We might end up having html strings as part of our text.

First, let us try to remove the HTML tags using regular expressions.

`Regex breakdown:`
```Python
'<.*?>'
```
* `<` and `>`: simply match the opening and closing brackets of HTML tags, e.g. \<div>
* `.*?`: This is the `non-greedy` or `lazy quantifier` *?, which matches any character (represented by `.` ) zero or more times, but it does so as few times as possible to make a valid match. In the context of HTML tags, this means it will match the shortest possible sequence of characters between the opening < and closing > tags.

So, the entire regular expression `'<.*?>'` is used to match and capture the shortest possible HTML tag found in a text string. This is useful in cases where you want to extract or remove HTML tags from a text while preserving the shortest possible tag structure.

In [ ]:
def remove_html(text :str) -> str:
    """
    Remove HTML tags and content from the input text.

    This function searches for HTML tags within the input text and removes them,
    leaving only the plain text content.

    Args:
        text (str): The input text containing HTML tags to be removed.

    Returns:
        str: A new string with HTML tags and content removed.

    Example:
        >>> remove_html("<p>This is <b>bold</b> text.</p>")
        "This is bold text."
    """
    # Your code here


text = """<div>
<h1> H2O</h1>
<SomeComponent/>
<p> AutoML</p>
<a href="https://www.h2o.ai/products/h2o-driverless-ai/"> Driverless AI</a>
</div>"""

print(remove_html('text'))

We can also use `BeautifulSoup` package to get the text from HTML document in a more elegant way.

In [ ]:
from bs4 import BeautifulSoup

def remove_html(text :str) -> str:
    """
    Remove HTML tags and content from the input text using BeautifulSoup.

    This function utilizes the BeautifulSoup library to parse the input text as HTML
    and then extracts and returns the plain text content, removing all HTML tags.

    Args:
        text (str): The input text containing HTML tags to be removed.

    Returns:
        str: A new string with HTML tags and content removed.

    Example:
        >>> remove_html("<p>This is <b>bold</b> text.</p>")
        "This is bold text."
    """
    # Your code here
    # check BeautifulSoup decoumentation

text = """<div>
<h1> H2O</h1>
<p> AutoML</p>
<a href="https://www.h2o.ai/products/h2o-driverless-ai/"> Driverless AI</a>
</div>
"""

print(remove_html(text))

**In your opinion which tool between regex or a parser is preferable and why? Write you thoughts**

*Here you answer*

## **SpaCy**
Similarly to NLTK, SpaCy is another popular NLP library with many features and pretrained models. It's extremely optimized and user-friendly.

Sometimes less flexible if compared to NLTK (which provides a wide range of algorithms) it has comparable performance, or slightly better in tokenization and POS-tagging, underperforming in sentence tokenization.

While NLTK can be seen as a large toolbox for NLP, SpaCy offers more ready-to-use solutions for production. Amoung the other funcionalities, there are:
- Tokenization
- Part-of-speech (POS) tagging
- Dependency Parsing
- Lemmatization
- Named Entity Recognition
- Sentence Boundary Detection
- Text Classification
- etc

It is worth it to that a look at it!

Let's import the library and download a pretrained model for the English language

In [ ]:
import spacy
!python -m spacy download en_core_web_sm

The SpaCy's Language model (named "nlp" in this code) contains all the components and data needed for the analysis. Calling it on a text string will return a Doc object

In [ ]:
nlp = spacy.load("en_core_web_sm") # pretrained model

doc = nlp("Apple is looking at buying U.K. startup for $1 billion")
for token in doc:
    print('{0}\t{1}\t{2}'.format(token.text, token.pos_, token.dep_))

As we have seen the first information we can get is the tokenization of the sequence with the corresponding tokens

In [ ]:
# Your code here:
# tokenize the sentence collecting the tokens with a list comprehension
# print the list of tokens


Tokens contain many attributes like the corresponding lemma, pos tag, syntax dependency, word shape (e.g. capitalization, punctuation, digits), is alpha, is a stop word

In [ ]:
for token in doc:
    string = '\t\t'.join([token.text, token.lemma_, token.pos_, token.tag_, token.dep_,
            token.shape_, str(token.is_alpha), str(token.is_stop)])
    print(string)

In [ ]:
# Your code here:
# remove stopwords using the information given by SpaCy
# return the result as a list of tokens

print(filtered)
assert(filtered == ['Apple', 'looking', 'buying', 'U.K.', 'startup', '$', '1', 'billion'])

SpaCy also provides a Named Entity Recognition (NER) engine recognizing categories such as person, country, organization etc.

Instead of iterating over the tokens, we rely on entities (ents) in this case

In [ ]:
for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

To conclude let's visualize the dependecy tree instead of printing the result on the terminal

In [ ]:
from spacy import displacy
displacy.serve(doc, style="dep", auto_select_port=True)

# keep in mind to stop the execution after running this cell: it's a service running on your PC

## **Subword Tokenization**

What we have seen so far was a traditional pre-processing pipeline still relevant in some tasks, however nowadays the standard approach for NLP rely on huge neural models that can perform feature extraction by themself, without human intervention. In this context all the information are needed for the model: punctuation, emojis, stopwords and even case sensitive words.

The only pre-processing steps required deal with data cleaning and sanitization, as well as tokenization. Pre-trained models usually have their own pre-trained tokenizer and sometimes it could be a good idea to pre-tokenize the input text, only once at the beginning, and work with sequences of tokens instead of strings.

Let's use therefore a modern subword tokenizer

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased") # import the pre-trained tokenizer from BERT, a model we will see in the last lectures

We are now using the HugginFace 🤗 library and in particular its tokenizers, with the tokenize() method we can directly see the effect of such tokenization

In [ ]:
tokenizer.tokenize("BERT is an encoder-only transformer architecture!")

Being it a subword tokenizer, words that do not appear in the (learned) vocabulary are splitted in subtokens, here identified with '##' at the beginning

In [ ]:
tokenizer.tokenize("transformer")

In [ ]:
tokenizer.tokenize("testing the tokenizer")

Keep in mind, though, that in this context the goal is to prepare the data for ML analysis using neural models, subtokens per se are not really usefull for humans but
Neural networks can only understand numbers, not string. As we have seen during the lecture the tokenizer will provide a numerical representation by converting tokens to word indeces (the corresponding index of the token in the vocabulary)

In [ ]:
tokenizer.encode('testing the tokenizer')

You might have notices that compared to the tokenizer() method we have now 6 tokens instead of 4, converting the ids back to string can help us understanding what is goind on

In [ ]:
input_ids = tokenizer.encode('testing the tokenizer')
tokenizer.decode(input_ids)

BERT's tokenizer automatically added two special tokens at the begging and ending of the sequence, respectively [CLS] and [SEP] or 101 and 102 in numberical values.

These tokens are used to delimit sentences and represent the entire sequence for tasks such as classification as we will see more in detail later on...

convert_ids_to_tokens, and the dual method convert_tokens_to_ids, can perform this conversion between tokens and ids and vice versa

In [ ]:
print(tokenizer.convert_ids_to_tokens(101))
print(tokenizer.convert_ids_to_tokens(1996))

Each word can therefore be splitted into one or more subtokens, depending on the tokenizer. In NLP "fertility" measures the average number of subwords produced per word by a tokenizer.

Let's compute the fertility on the brown corpus: a pretokenized collection of texts already available on NLTK

In [ ]:
import nltk.corpus
nltk.download('brown')
brown = nltk.corpus.brown
words = brown.words()

print(words[:14])
print('Number of words: ' + str(len(words)))

In [ ]:
# Here your code
# compute the fertility on the brown corpus using BERT's pre-trained tokenizer

print(fertility)

**We computed the fertility on the Brown corpus (English texts) using a tokenizer trained for English, what if we used instead a multilingual model or a tokenizer trained in another language? Based on you understanding of the algorithm, what do you expect and why?**

*Here you answer*

## +++ End of the mandatory section +++

## **Advance only**

To get to the `advanced` grade try to put together what you have learned into a script and process the examples provided in `spam.tsv`. The final goal is to discriminate between spam and not spam using a random forest classifier. Adapt therefore the pre-processing pipeline to this need (e.g. removing or non removing stopwords? Punctuation? Stemming or Lemmatization? Other things to clean?). Please justify and comment on these decisions. Feel free to add as many steps as you feel necessary, as long as you justify the choice.

Complete the `spam.py` script and try to run the classification process. How your decisions impacted the performance (pay attention to the F1-score)? Try different strategies and report what you have found, what are your explanations?